In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [7]:
train_datagen = ImageDataGenerator(
        rescale=1./255,
        zoom_range=0.2,
        horizontal_flip=True,
         validation_split=0.2)
training_set = train_datagen.flow_from_directory(
        '../data/chest_xray/train',
        target_size=(224, 224),
        batch_size=32,
        class_mode='binary',
        subset='training')

Found 4173 images belonging to 2 classes.


Test set prep

In [8]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory(
        '../data/chest_xray/test',
        target_size=(64, 64),
        batch_size=32,
        class_mode='binary',
        shuffle=False)

Found 624 images belonging to 2 classes.


Validation set prep

In [9]:
validation_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

validation_set = validation_datagen.flow_from_directory(
    '../data/chest_xray/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 1043 images belonging to 2 classes.


In [10]:
print("Class mapping:", training_set.class_indices)

Class mapping: {'NORMAL': 0, 'PNEUMONIA': 1}


In [11]:
images, labels = next(training_set)

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Pixel range:", images.min(), images.max())

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Pixel range: 0.0 1.0


BUILDING CNN

In [12]:
cnn=tf.keras.models.Sequential()

In [13]:
cnn.add(tf.keras.layers.Conv2D(
    filters=32,
    kernel_size=3,
    activation='relu',
    input_shape=[224, 224, 3]
))

c:\Users\Srijan Yadav\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
cnn.add(tf.keras.layers.MaxPool2D(
    pool_size=2,
    strides=2
))

In [15]:
cnn.add(tf.keras.layers.Conv2D(
    filters=64,
    kernel_size=3,
    activation='relu'
))

cnn.add(tf.keras.layers.MaxPool2D(
    pool_size=2,
    strides=2
))

In [16]:
cnn.add(tf.keras.layers.Conv2D(
    filters=128,
    kernel_size=3,
    activation='relu'
))

cnn.add(tf.keras.layers.MaxPool2D(
    pool_size=2,
    strides=2
))

In [17]:
cnn.add(tf.keras.layers.GlobalAveragePooling2D())

Dense layer

In [18]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

In [19]:

cnn.add(tf.keras.layers.Dropout(0.5))

cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [20]:
cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [23]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,889 (429.25 KB)

 Trainable params: 109,889 (429.25 KB)

 Non-trainable params: 0 (0.00 B)

BALANCING

In [24]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(training_set.classes),
    y=training_set.classes
)

class_weights = dict(enumerate(class_weights))

print("Class indices:", training_set.class_indices)
print("Class weights:", class_weights)

Class indices: {'NORMAL': 0, 'PNEUMONIA': 1}
Class weights: {0: np.float64(1.9445479962721341), 1: np.float64(0.6730645161290323)}


In [25]:
history = cnn.fit(
    x=training_set,
    validation_data=validation_set,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 174s 1s/step - accuracy: 0.5557 - loss: 0.6816 - val_accuracy: 0.7114 - val_loss: 0.6150
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 68s 521ms/step - accuracy: 0.7100 - loss: 0.5743 - val_accuracy: 0.6692 - val_loss: 0.6225
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 91s 692ms/step - accuracy: 0.7855 - loss: 0.4681 - val_accuracy: 0.7958 - val_loss: 0.4341
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 68s 517ms/step - accuracy: 0.8241 - loss: 0.3872 - val_accuracy: 0.7613 - val_loss: 0.4749
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 67s 508ms/step - accuracy: 0.8200 - loss: 0.3872 - val_accuracy: 0.8408 - val_loss: 0.3813
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 66s 504ms/step - accuracy: 0.8416 - loss: 0.3543 - val_accuracy: 0.8591 - val_loss: 0.3309
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 66s 502ms/step - accuracy: 0.8459 - loss: 0.3532 - val_accuracy: 0.7287 - val_loss: 0.5676
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 72s 553ms/step - accuracy: 0.8497 - loss: 0.3